In [ ]:
# %% imports and environment setup
from importlib import import_module
from pathlib import Path
import sys, os, multiprocessing as mp
import imageio_ffmpeg
import torch
from torch._functorch._aot_autograd.logging_utils import model_name

# 🧠 use all logical cores efficiently
LOGICAL = mp.cpu_count() or 8
os.environ["OMP_NUM_THREADS"] = str(LOGICAL)
os.environ["MKL_NUM_THREADS"] = str(LOGICAL)
os.environ["OPENBLAS_NUM_THREADS"] = str(LOGICAL)
os.environ["NUMEXPR_NUM_THREADS"] = str(LOGICAL)

torch.set_num_threads(LOGICAL // 2)
torch.set_num_interop_threads(LOGICAL // 2)

# ✅ detect Intel oneDNN (IPEX) if available
try:
    import intel_extension_for_pytorch as ipex
    HAS_IPEX = True
    print("[info] Intel Extension for PyTorch (IPEX) detected.")
except ImportError:
    HAS_IPEX = False
    print("[info] IPEX not found, running standard PyTorch.")

# add repo root
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# import internal tools
input_selector = import_module("tools.input_selector")
ffmpeg_tools = import_module("tools.FFmpeg.FFmpeg_utilization")
model_selector = import_module("tools.model_selector")
report = import_module("tools.preview_report")

# check hardware
print(f"[cpu] threads: {LOGICAL}")
print(f"[gpu] available: {torch.cuda.is_available()}")


[info] IPEX not found, running standard PyTorch.
[cpu] threads: 20
[gpu] available: False


In [ ]:
# %% [markdown]
# ## 2️⃣ Input: Select video
# Either type a path manually or assign one directly for faster testing.

# %%
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

input_selector = import_module("tools.input_selector")
video_path = input_selector.get_input_video_path(allowed_exts=VIDEO_EXTS)

# Example: hard-code during testing
# video_path = ROOT / "demoGreyscale.mp4"

print(f"[ok] selected video: {video_path}")


Enter path to input video:
[ok] selected video: C:\Users\bebo\Documents\GitHub\VideoRasterization\demoGreyscale.mp4


In [ ]:
# %% [markdown]
# ## 3️⃣ Extract frames using FFmpeg
# Uses your tools/FFmpeg/FFmpeg_utilization.py

# %%
temp_root = ROOT / "temp"
temp_root.mkdir(parents=True, exist_ok=True)

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_tools = import_module("tools.FFmpeg.FFmpeg_utilization")

frames_dir = ffmpeg_tools.extract_frames(ffmpeg_path, video_path, temp_root)

if not frames_dir or not Path(frames_dir).exists():
    raise RuntimeError("Frame extraction failed.")
else:
    print(f"[ok] extracted frames dir: {frames_dir}")


Choose extraction mode:
1 = Full Quality (PNG, large size)
2 = Faster Encoding (JPG, smaller size)
[mode] faster JPG extraction selected.
[extract] saving frames to: C:\Users\bebo\Documents\GitHub\VideoRasterization\temp\20251016_235419\frames
[ok] frame extraction complete.
[ok] extracted frames dir: C:\Users\bebo\Documents\GitHub\VideoRasterization\temp\20251016_235419\frames


In [ ]:
# %% 4️⃣ prepare output + run model
frames_path = Path(frames_dir)
color_dir = frames_path.parent / f"{frames_path.name}_colorized"
color_dir.mkdir(parents=True, exist_ok=True)

use_gpu = torch.cuda.is_available()

# optional variant (only used by Zhang)
try:
    zhang_variant
except NameError:
    zhang_variant = None

if HAS_IPEX and not use_gpu:
    print("[info] enabling Intel oneDNN optimizations...")
    ipex.enable_onednn_fusion(True)
    try:
        ipex.set_fp32_math_mode(mode="BF16")
    except Exception:
        pass

# auto batch sizing
batch_size = max(1, LOGICAL // 2)

print(f"[start] running model: {model_name} (batch={batch_size})")
model_selector.run_colorizer(
    model_name="colorize_zhang",
    frames_dir=frames_path,
    color_dir=color_dir,
    models_dir=ROOT / "models",
    zhang_variant=zhang_variant,
    preview=False,
    use_gpu=use_gpu,
    batch_size=batch_size,
    num_threads=LOGICAL,
    input_size=224,
    progress=True,
    prefetch_workers=LOGICAL // 4,
    save_workers=2,
)
print(f"[ok] colorization complete: {color_dir}")


[start] running model: model (batch=10)
[start] zhang=None | frames=828 | batch=10 | threads=20 | prefetch=5 | save_workers=2
[ok] model ready in 0.2s


In [ ]:
# %% 5️⃣ Temporal Smoothing Step
from tools.TemporalSmoothing import apply_temporal_smoothing

# Ask for smoothing parameters
try:
    smooth_dir
except NameError:
    smooth_dir = None

if not smooth_dir:
    smooth_dir = input("Enter output folder for smoothed frames (blank = auto): ").strip() or (
        Path(color_dir).parent / f"{Path(color_dir).name}_TemporalSmoothed"
    )

try:
    window_size = int(input("Enter temporal smoothing window (odd number, default=9): ") or 9)
    if window_size % 2 == 0:
        window_size -= 1
    if window_size < 3:
        window_size = 3
except Exception:
    window_size = 9

apply_temporal_smoothing(
    input_folder=color_dir,
    output_folder=smooth_dir,
    use_onnx=True,
    window_size=window_size,
)

print(f"[ok] temporal smoothing complete: {smooth_dir}")


In [ ]:
# %% [markdown]
# ## 6️⃣ (Optional) Rebuild video or generate report
# You can add temporal smoothing or report steps later here.

# %%
# example placeholders:
# ffmpeg_tools.rebuild_video(...)
# report = import_module("tools.preview_report")
# report.generate_report(...)

print("[done] pipeline finished (frames ready).")

[done] pipeline finished (frames ready).
